# Zero-Shot Traffic Classification for Unseen Applications Using Convolutional Neural Network and Word2Vec

**Authors:** Prerna Awati, Aishwarya A, Nishchita N, Sonu Vaishnavi, Uday Kulkarni, Veena Hunasimarad  
**Affiliation:** School of Computer Science and Engineering, KLE Technological University, Hubballi, India  

---

## 1. Project Overview
Modern network traffic is increasingly protected by encryption protocols (TLS 1.3, QUIC, and VPN tunneling), preventing deep packet payload inspection. Traditional supervised network classifiers fail whenever new, unobserved applications emerge on the network.

This project implements an end-to-end **Metric Learning and Zero-Shot Classification Framework** combining:
1. **Symbolic Feature Quantization:** Continuous flow statistics (e.g. packet length, inter-arrival time) are converted into symbolic categorical tokens (`_low`, `_medium`, `_high`, `_very_high`).
2. **Word2Vec Semantic Embedding:** Discretized flow tokens are projected into a continuous semantic vector space via CBOW.
3. **1D-ResNet Encoder:** Deep 1D residual convolutional neural network transforms flow embeddings into a discriminative latent metric space.
4. **Prototypical Networks:** Class prototypes (centroids) are computed geometrically, enabling nearest-centroid metric classification and unseen application discovery.


## 2. Setup and Dependencies
Import core libraries and configure the computation device.

In [ ]:
import os
import zipfile
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.io import arff

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[✓] Computation Device: {DEVICE}')


## 3. Dataset Ingestion (ISCX VPN-nonVPN Scenario-B)
Loads the ARFF dataset containing 23 flow-level statistical features across 7 application categories:
- **Seen Classes:** `CHAT`, `MAIL`, `FT`, `P2P`
- **Unseen Classes:** `BROWSING`, `STREAMING`, `VOIP`

In [ ]:
def find_arff_dataset():
    candidates = [
        '../data/Scenario B-ARFF/TimeBasedFeatures-Dataset-120s-AllinOne.arff',
        'data/Scenario B-ARFF/TimeBasedFeatures-Dataset-120s-AllinOne.arff',
        '../data/Scenario B-ARFF.zip',
        'data/Scenario B-ARFF.zip',
        '/content/Scenario B-ARFF.zip'
    ]
    for path in candidates:
        if os.path.exists(path):
            if path.endswith('.zip'):
                target_dir = 'data'
                os.makedirs(target_dir, exist_ok=True)
                with zipfile.ZipFile(path, 'r') as zf:
                    zf.extractall(target_dir)
                for root, _, files in os.walk(target_dir):
                    for f in files:
                        if f.endswith('.arff'):
                            return os.path.join(root, f)
            elif path.endswith('.arff'):
                return path
    return None

arff_file = find_arff_dataset()
if arff_file and os.path.exists(arff_file):
    print(f'[✓] Loaded dataset from: {arff_file}')
    data, meta = arff.loadarff(arff_file)
    raw_df = pd.DataFrame(data)
    for col in raw_df.columns:
        if raw_df[col].dtype == 'object':
            raw_df[col] = raw_df[col].apply(lambda x: x.decode('utf-8') if isinstance(x, bytes) else str(x))
    raw_df.replace([np.inf, -np.inf], np.nan, inplace=True)
    raw_df.fillna(0, inplace=True)
    print(f'[✓] Loaded {raw_df.shape[0]} flow records and {raw_df.shape[1]} columns.')
else:
    print('[!] Place TimeBasedFeatures-Dataset-120s-AllinOne.arff in the data/ folder.')


## 4. Feature Quantization (Algorithm 1 in Paper)
Continuous numerical features are binned into categorical tokens (`_low`, `_medium`, `_high`, `_very_high`) to form symbolic 'sentences'.

In [ ]:
LABEL_COLUMN = 'class1'

def quantize_features(df, label_col, n_bins=4):
    quantized = pd.DataFrame()
    features = [c for c in df.columns if c != label_col]
    bin_labels = ['_low', '_medium', '_high', '_very_high']
    for col in features:
        try:
            if df[col].nunique() <= 1:
                quantized[col] = f'{col}_const'
            else:
                quantized[col] = pd.qcut(df[col], q=n_bins, labels=bin_labels, duplicates='drop')
                quantized[col] = col + quantized[col].astype(str)
        except Exception:
            try:
                quantized[col] = pd.cut(df[col], bins=n_bins, labels=bin_labels, duplicates='drop')
                quantized[col] = col + quantized[col].astype(str)
            except Exception:
                quantized[col] = f'{col}_const'
    quantized[label_col] = df[label_col].values
    return quantized

if 'raw_df' in locals():
    df_quantized = quantize_features(raw_df, LABEL_COLUMN)
    print('[✓] Quantization complete. Sample tokens:')
    print(df_quantized.iloc[:3, :5])


## 5. Data Partitioning and Split
We target the 7 application categories evaluated in the paper and split the data into 80% train and 20% test sets.

In [ ]:
SEEN_APPS = ['CHAT', 'MAIL', 'FT', 'P2P']
UNSEEN_APPS = ['BROWSING', 'STREAMING', 'VOIP']
ALL_APPS = SEEN_APPS + UNSEEN_APPS

if 'df_quantized' in locals():
    data_df = df_quantized[df_quantized[LABEL_COLUMN].isin(ALL_APPS)].copy()
    print('[✓] Label distribution:')
    print(data_df[LABEL_COLUMN].value_counts())
    
    train_df, test_df = train_test_split(
        data_df,
        test_size=0.2,
        train_size=0.8,
        stratify=data_df[LABEL_COLUMN],
        random_state=42
    )
    print(f'\n[✓] Training split: {len(train_df)} | Test split: {len(test_df)}')


## 6. Word2Vec Semantic Feature Representation
Trains a Continuous Bag-of-Words (CBOW) model over the tokenized flow records to learn 100-dimensional semantic embeddings.

In [ ]:
W2V_DIM = 100
W2V_WINDOW = 5
W2V_EPOCHS = 50

if 'train_df' in locals():
    feat_cols = [c for c in train_df.columns if c != LABEL_COLUMN]
    sentences = [[str(v) for v in row] for row in train_df[feat_cols].values]
    
    w2v = Word2Vec(sentences, vector_size=W2V_DIM, window=W2V_WINDOW, min_count=1, workers=4, epochs=W2V_EPOCHS, sg=0, seed=42)
    print(f'[✓] Word2Vec trained. Vocabulary: {len(w2v.wv)} tokens')
    
    def embed_flows(df, model, label_col):
        cols = [c for c in df.columns if c != label_col]
        X, y = [], []
        for _, r in df.iterrows():
            toks = [str(v) for v in r[cols].values]
            vecs = [model.wv[t] for t in toks if t in model.wv]
            X.append(np.mean(vecs, axis=0) if vecs else np.zeros(W2V_DIM))
            y.append(r[label_col])
        return np.array(X, dtype=np.float32), np.array(y)
    
    X_train, y_train_str = embed_flows(train_df, w2v, LABEL_COLUMN)
    X_test, y_test_str = embed_flows(test_df, w2v, LABEL_COLUMN)
    
    le = LabelEncoder()
    y_train = le.fit_transform(y_train_str)
    y_test = le.transform(y_test_str)
    class_names = list(le.classes_)
    n_classes = len(class_names)
    print(f'[✓] Encoded classes: {class_names}')


## 7. Model Architecture (1D-ResNet & Prototypical Loss)
Defines the deep 1D residual convolutional neural network and the Prototypical Network loss formulation.

In [ ]:
class ResidualBlock1D(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, stride=stride, padding=kernel_size//2, bias=False)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, stride=1, padding=kernel_size//2, bias=False)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(out_channels)
            )

    def forward(self, x):
        res = x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(res)
        return F.relu(out)

class ResNet1D(nn.Module):
    def __init__(self, input_dim=100, hidden_dim=128, output_dim=64):
        super().__init__()
        self.conv1 = nn.Conv1d(1, hidden_dim, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.maxpool = nn.MaxPool1d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self._make_layer(hidden_dim, hidden_dim, 2)
        self.layer2 = self._make_layer(hidden_dim, hidden_dim*2, 2, stride=2)
        self.layer3 = self._make_layer(hidden_dim*2, hidden_dim*4, 2, stride=2)
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(hidden_dim*4, output_dim)

    def _make_layer(self, in_c, out_c, blocks, stride=1):
        layers = [ResidualBlock1D(in_c, out_c, stride=stride)]
        for _ in range(1, blocks):
            layers.append(ResidualBlock1D(out_c, out_c))
        return nn.Sequential(*layers)

    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.maxpool(out)
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.avgpool(out)
        out = out.view(out.size(0), -1)
        return self.fc(out)

def euclidean_distance(x, y):
    n, m, d = x.size(0), y.size(0), x.size(1)
    return torch.pow(x.unsqueeze(1).expand(n, m, d) - y.unsqueeze(0).expand(n, m, d), 2).sum(2)

def compute_prototypes(embeddings, labels, n_classes):
    prototypes = torch.zeros(n_classes, embeddings.size(1), device=embeddings.device)
    for i in range(n_classes):
        mask = labels == i
        if mask.sum() > 0:
            prototypes[i] = embeddings[mask].mean(0)
    return prototypes

def prototypical_loss(embeddings, labels, n_support, n_classes):
    support_idx = torch.zeros(len(labels), dtype=torch.bool, device=labels.device)
    for i in range(n_classes):
        idx = torch.where(labels == i)[0]
        if len(idx) >= n_support:
            support_idx[idx[:n_support]] = True
    
    if support_idx.sum() < n_classes:
        protos = compute_prototypes(embeddings, labels, n_classes)
        dists = euclidean_distance(embeddings, protos)
        log_p = F.log_softmax(-dists, dim=1)
        loss = F.nll_loss(log_p, labels)
        _, preds = log_p.max(1)
        return loss, (preds == labels).float().mean(), protos
    
    protos = compute_prototypes(embeddings[support_idx], labels[support_idx], n_classes)
    dists = euclidean_distance(embeddings[~support_idx], protos)
    log_p = F.log_softmax(-dists, dim=1)
    loss = F.nll_loss(log_p, labels[~support_idx])
    _, preds = log_p.max(1)
    return loss, (preds == labels[~support_idx]).float().mean(), protos


## 8. Training Pipeline
Trains the ResNet-1D encoder using episodic Prototypical Loss.

In [ ]:
class TrafficDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

if 'X_train' in locals():
    BATCH_SIZE = 32
    LR = 0.001
    EPOCHS = 15
    N_SUPPORT = 5
    
    train_loader = DataLoader(TrafficDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
    
    model = ResNet1D(input_dim=W2V_DIM, hidden_dim=128, output_dim=64).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    
    total_p = sum(p.numel() for p in model.parameters())
    print(f'[✓] Trainable ResNet-1D parameters: {total_p:,}')
    
    model.train()
    for epoch in range(EPOCHS):
        ep_loss, ep_acc, count = 0.0, 0.0, 0
        for bx, by in train_loader:
            bx, by = bx.to(DEVICE), by.to(DEVICE)
            emb = model(bx)
            loss, acc, _ = prototypical_loss(emb, by, N_SUPPORT, n_classes)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            ep_loss += loss.item()
            ep_acc += acc.item()
            count += 1
        if (epoch + 1) % 5 == 0 or epoch == EPOCHS - 1:
            print(f'Epoch [{epoch+1:2d}/{EPOCHS:2d}] -> Loss: {ep_loss/count:.4f} | Batch Acc: {(ep_acc/count)*100:.2f}%')


## 9. Evaluation and Confusion Matrix (Paper Benchmark)
Evaluates performance on the held-out test split and plots the confusion matrix heatmap corresponding to **Figure 6** of the paper.

In [ ]:
if 'model' in locals():
    model.eval()
    with torch.no_grad():
        train_emb = model(torch.FloatTensor(X_train).to(DEVICE))
        prototypes = compute_prototypes(train_emb, torch.LongTensor(y_train).to(DEVICE), n_classes)
        
        test_emb = model(torch.FloatTensor(X_test).to(DEVICE))
        dists = euclidean_distance(test_emb, prototypes)
        _, preds = dists.min(1)
        preds_np = preds.cpu().numpy()
    
    test_acc = accuracy_score(y_test, preds_np)
    print(f'\n{"="*70}')
    print(f'TEST ACCURACY: {test_acc*100:.2f}%')
    print(f'{"="*70}\n')
    
    print('--- Classification Report ---')
    print(classification_report(y_test, preds_np, target_names=class_names, zero_division=0))
    
    cm = confusion_matrix(y_test, preds_np)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted Application')
    plt.ylabel('True Application')
    plt.title('Confusion Matrix Heatmap (ISCX Scenario-B)')
    plt.tight_layout()
    plt.show()


## 10. Application Similarity & Proximity Analysis
Examines the proximity of unseen application flows (`BROWSING`, `STREAMING`, `VOIP`) to seen prototypes.

In [ ]:
if 'preds_np' in locals():
    print('\n' + '='*70)
    print('APPLICATION SIMILARITY MAPPING')
    print('='*70)
    
    for app in UNSEEN_APPS:
        mask = y_test_str == app
        if mask.sum() > 0:
            mapped = [class_names[p] for p in preds_np[mask]]
            names, counts = np.unique(mapped, return_counts=True)
            print(f'\nTarget Unseen: {app} ({mask.sum()} samples)')
            for n, c in sorted(zip(names, counts), key=lambda x: -x[1]):
                print(f'  -> {n:10s}: {c:4d} ({(c/mask.sum())*100:5.1f}%)')
